# 00 — Data Preprocessing (M5, scalable)

Builds the long-format demand panel that every downstream notebook consumes. It
loads the raw M5 files, selects a configurable subset (a few hundred series for a
quick CPU run, up to the full 30,490-series panel on a GPU box), melts the wide
daily-sales matrix into a long `(series, date)` panel, joins the calendar (events,
SNAP) and weekly sell-prices, engineers lag / rolling / calendar / price features,
and writes a compact parquet panel plus per-series metadata and the WRMSSE scaling
weights.

**Inputs:** `data/raw_m5/{sales_train_evaluation,calendar,sell_prices}.csv`
**Outputs:** `data/panel.parquet`, `data/series_meta.parquet`

In [25]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import features as F

RAW = ROOT / "data" / "raw_m5"
OUT = ROOT / "data"
OUT.mkdir(exist_ok=True)

# --- scale control -------------------------------------------------------
# SUBSET=None uses every series. An int keeps the top-N highest-volume series.
# CATEGORIES filters M5 product categories: {'HOBBIES','HOUSEHOLD','FOODS'}.
SUBSET = 10000            # set to None for the full M5 panel (GPU recommended)
CATEGORIES = None       # e.g. ['FOODS'] to focus a category; None = all
START_DATE = "2013-01-01"
print(f"config: SUBSET={SUBSET}  CATEGORIES={CATEGORIES}  from {START_DATE}")

config: SUBSET=10000  CATEGORIES=None  from 2013-01-01


Load the three raw M5 tables: wide daily sales, the calendar, and weekly sell-prices.

In [26]:
sales = pd.read_csv(RAW / "sales_train_evaluation.csv")
calendar = pd.read_csv(RAW / "calendar.csv", parse_dates=["date"])
prices = pd.read_csv(RAW / "sell_prices.csv")

# some M5 calendar exports drop the d_1..d_N column; rebuild it from date order
if "d" not in calendar.columns:
    calendar = calendar.sort_values("date").reset_index(drop=True)
    calendar["d"] = ["d_" + str(i + 1) for i in range(len(calendar))]
print("sales:", sales.shape, "| calendar:", calendar.shape, "| prices:", prices.shape)
sales.iloc[:2, :8]

sales: (30490, 1947) | calendar: (1969, 14) | prices: (6841121, 4)


,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0


Optionally filter to chosen categories, then keep the top-N highest-volume series so the notebook scales from laptop to full panel.

In [27]:
if CATEGORIES:
    sales = sales[sales["cat_id"].isin(CATEGORIES)].copy()

day_cols = [c for c in sales.columns if c.startswith("d_")]
sales["_vol"] = sales[day_cols].sum(axis=1)
if SUBSET is not None:
    sales = sales.sort_values("_vol", ascending=False).head(SUBSET).copy()
sales = sales.drop(columns="_vol")
sales["series_id"] = sales["item_id"] + "--" + sales["store_id"]
print(f"selected {len(sales):,} series across "
      f"{sales['store_id'].nunique()} stores, {sales['cat_id'].nunique()} categories")

selected 10,000 series across 10 stores, 3 categories


Melt the wide daily matrix into a long `(series, day)` panel and attach the calendar date for each `d_` column.

In [28]:
id_cols = ["series_id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
long = sales.melt(id_vars=id_cols, value_vars=day_cols,
                  var_name="d", value_name="units")
cal_small = calendar[["d", "date", "wm_yr_wk", "event_type_1",
                      "snap_CA", "snap_TX", "snap_WI"]]
long = long.merge(cal_small, on="d", how="left")
long = long[long["date"] >= pd.Timestamp(START_DATE)].copy()
print("long panel:", long.shape, "| dates", long["date"].min().date(), "→", long["date"].max().date())

long panel: (12380000, 14) | dates 2013-01-01 → 2016-05-22


Attach the weekly sell-price for each (store, item, week); the per-row SNAP flag is selected by the series' state.

In [29]:
long = long.merge(prices, on=["store_id", "item_id", "wm_yr_wk"], how="left")
snap_map = {"CA": "snap_CA", "TX": "snap_TX", "WI": "snap_WI"}
long["snap"] = 0
for st, col in snap_map.items():
    m = long["state_id"] == st
    long.loc[m, "snap"] = long.loc[m, col].fillna(0).astype(int)
long = long.drop(columns=["snap_CA", "snap_TX", "snap_WI", "d", "wm_yr_wk"])
long["event_type_1"] = long["event_type_1"].fillna("none")
for ev in ["Cultural", "National", "Religious", "Sporting", "none"]:
    long[f"ev_{ev}"] = (long["event_type_1"] == ev).astype(int)

Engineer demand lags, rolling statistics, calendar signals, and price-dynamics features using the shared `src.features` module.

In [30]:
long = F.add_calendar_features(long)
long = F.add_price_features(long, group_col="series_id")
long = F.add_lag_features(long, group_col="series_id", target="units")
long = F.encode_ids(long)
long = F.downcast(long)
print("panel with features:", long.shape)
print("feature columns:", [c for c in long.columns if c not in id_cols][:25], "...")

panel with features: (12380000, 47)
feature columns: ['units', 'date', 'event_type_1', 'sell_price', 'snap', 'ev_Cultural', 'ev_National', 'ev_Religious', 'ev_Sporting', 'ev_none', 'day_of_week', 'is_weekend', 'day_of_month', 'week_of_year', 'month', 'quarter', 'is_month_end', 'is_payday_window', 'price_lag_7', 'price_mean_28', 'price_ratio', 'price_change', 'on_promo', 'revenue', 'lag_1'] ...


Build per-series metadata (category ids and demand summary statistics) used by the segmentation and evaluation notebooks.

In [31]:
meta = (long.groupby("series_id")
        .agg(item_id=("item_id","first"), dept_id=("dept_id","first"),
             cat_id=("cat_id","first"), store_id=("store_id","first"),
             state_id=("state_id","first"),
             mean_units=("units","mean"), std_units=("units","std"),
             zero_frac=("units", lambda s: (s==0).mean()),
             total_rev=("revenue","sum"))
        .reset_index())
meta["cv"] = meta["std_units"] / meta["mean_units"].replace(0, np.nan)
print("series_meta:", meta.shape)
meta.head(3)

series_meta: (10000, 11)


,series_id,item_id,dept_id,cat_id,store_id,state_id,mean_units,std_units,zero_frac,total_rev,cv
0,FOODS_1_001--CA_2,FOODS_1_001,FOODS_1,FOODS,CA_2,CA,0.949919,1.531374,0.535541,2634.239990,1.612109
1,FOODS_1_001--CA_3,FOODS_1_001,FOODS_1,FOODS,CA_3,CA,1.005654,2.348904,0.626817,2785.919922,2.335697
2,FOODS_1_003--CA_1,FOODS_1_003,FOODS_1,FOODS,CA_1,CA,0.758481,1.093372,0.552504,2833.469971,1.441527


Write the panel and metadata to parquet for fast, typed loading downstream.

In [32]:
panel_path = OUT / "panel.parquet"
meta_path = OUT / "series_meta.parquet"
long.to_parquet(panel_path, index=False)
meta.to_parquet(meta_path, index=False)
print(f"saved → {panel_path}  ({panel_path.stat().st_size/1e6:.1f} MB, {len(long):,} rows)")
print(f"saved → {meta_path}   ({len(meta):,} series)")

saved → /Users/jonathan/Desktop/COWORK HOMEBASE/02 Projects/Demand Forecasting & Inventory RL/data/panel.parquet  (202.0 MB, 12,380,000 rows)
saved → /Users/jonathan/Desktop/COWORK HOMEBASE/02 Projects/Demand Forecasting & Inventory RL/data/series_meta.parquet   (10,000 series)
